In [ ]:
import os
import duckdb
from scipy import stats

# data set information

# setup paths and load data
drive_data_dir = "/content/drive/MyDrive/data"
parquet_path = os.path.join(drive_data_dir, "cleaned_patent_panel.parquet")

con = duckdb.connect()

df = con.execute(f"""
    SELECT
        patent_id,
        vc_backed,
        grant_year,
        normalized_forward_citations,
        npl_ratio,
        total_claims,
        LN(total_claims + 1) AS log_claims
    FROM '{parquet_path}'
    WHERE grant_year BETWEEN 2000 AND 2020;
""").df()

# dataset summary and variable definitions
print("# dataset summary and variable definitions\n")
print(f"# total observations: {len(df):,} patents")
print(f"# time period: {df['grant_year'].min()} - {df['grant_year'].max()} grant years")
print("# source: uspto database matched with venture capital records\n")

print("# variables")
print("# patent_id: unique patent identifier")
print("# vc_backed: 1 if assignee received venture capital, 0 otherwise")
print("# grant_year: calendar year patent was granted")
print("# normalized_forward_citations: focal citations divided by mean of cpc subclass and grant year cohort")
print("# npl_ratio: non-patent literature / (backward citations + non-patent literature)")
print("# total_claims: count of patent claims")
print("# log_claims: ln(total_claims + 1)\n")

# summary statistics
stats_cols = ['normalized_forward_citations', 'npl_ratio', 'total_claims']
summary = df[stats_cols].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]

print("# summary statistics")
print("# variable                      count    mean  std dev   min   25%   50%   75%     max")
for var, row in summary.iterrows():
    print(
        f"# {var:<28} "
        f"{row['count']:8.2f} "
        f"{row['mean']:5.2f} "
        f"{row['std']:8.2f} "
        f"{row['min']:5.2f} "
        f"{row['25%']:5.2f} "
        f"{row['50%']:5.2f} "
        f"{row['75%']:5.2f} "
        f"{row['max']:7.2f}"
    )

# group mean comparisons (welch's t-test)
print("\n# group mean comparisons (welch's t-test)")

vc_group = df[df['vc_backed'] == 1]
non_vc_group = df[df['vc_backed'] == 0]

for var in ['normalized_forward_citations', 'total_claims', 'npl_ratio']:
    vc_clean = vc_group[var].dropna()
    non_vc_clean = non_vc_group[var].dropna()

    t_stat, p_val = stats.ttest_ind(vc_clean, non_vc_clean, equal_var=False)

    p_format = f"{p_val:.4f}" if p_val >= 0.001 else f"{p_val:.2e}"

    print(f"\n# {var}")
    print(f"# vc-backed mean: {vc_clean.mean():.3f} (n = {len(vc_clean)})")
    print(f"# non-vc mean:    {non_vc_clean.mean():.3f} (n = {len(non_vc_clean)})")
    print(f"# t-statistic:    {t_stat:.2f}")
    print(f"# p-value:        {p_format}")